## **#06. Recurrent Neural Networks and Text Mining (optional)**
- Instructor: Jaeung Sim (University of Connecticut)
- Course: OPIM 5671 Data Mining and Time Series Forecasting
- Last updated on November 3, 2026

This optional lab retains the original IMDb RNN exercise. The main changes separate validation from test data, add a majority-class baseline, correct accuracy for logits, and bound training cost. The stacked LSTM remains optional. No Transformer training is required.

Use Google Colab, preferably with a GPU. Run the single-LSTM section first. You may skip Section D and proceed to the final test cell. The setup installs Hugging Face `datasets` if needed. TensorFlow is included in standard Colab runtimes.

**Data source update:** This version loads raw IMDb reviews from the public [Stanford NLP mirror on Hugging Face](https://huggingface.co/datasets/stanfordnlp/imdb), avoiding TensorFlow Datasets' direct Stanford archive download. No Hugging Face login or Kaggle credentials are required for this public dataset. The official 25,000-review training and test splits are preserved. Training/validation membership changes relative to the earlier loader because the providers order the reviews differently, so rerun all cells and do not combine scores from the two versions.

**Objectives**
* Build a natural language processing (NLP) model with recurrent neural networks (RNNs) to classify texts.

**References**
* [Deep Learning Basics by Google Colab](https://colab.research.google.com/github/lexfridman/mit-deep-learning/blob/master/tutorial_deep_learning_basics/deep_learning_basics.ipynb)
* [Text classification with an RNN by TensorFlow](https://www.tensorflow.org/text/tutorials/text_classification_rnn)
* [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/)
* [Masking and padding with Keras by TensorFlow](https://www.tensorflow.org/guide/keras/masking_and_padding)

#### **Text Classification of IMDb Movie Reviews with RNNs**

In this practice, you will train a RNN model on the [IMDb large movie review dataset](https://ai.stanford.edu/~amaas/data/sentiment/) for sentiment analysis.

This dataset is constructed for binary sentiment classification and provides a set of 25,000 highly polar movie reviews for training, and 25,000 for testing. There is additional unlabeled data for use as well. Raw text and already processed bag of words formats are provided.

##### **1. Setup**

In [ ]:
# Import libraries and seed the experiment.
# Install only the new data-loading dependency when it is absent.
import importlib.util
import subprocess
import sys
if importlib.util.find_spec('datasets') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'])

import numpy as np
import tensorflow as tf
from datasets import load_dataset

SEED = 5671
tf.keras.utils.set_random_seed(SEED)

In [ ]:
# Create a helper function to plot graphs using `matplotlib`
import matplotlib.pyplot as plt

def plot_graphs(history, metric):
  plt.plot(history.history[metric])
  plt.plot(history.history['val_'+metric], '')
  plt.xlabel("Epochs")
  plt.ylabel(metric)
  plt.legend([metric, 'val_'+metric])

The IMDb dataset has binary labels: 0 = negative and 1 = positive. Load the original train/test splits from Stanford NLP's public Hugging Face mirror. The mirror supplies raw text, matching the input expected by our `TextVectorization` layer. The first run downloads and caches the data; later runs reuse that cache.

In [ ]:
# Load the official labeled splits from the public Stanford NLP mirror.
# Request only train and test files; do not download the unused unsupervised split.
raw_train, raw_test = load_dataset(
    'stanfordnlp/imdb',
    data_files={
        'train': 'plain_text/train-00000-of-00001.parquet',
        'test': 'plain_text/test-00000-of-00001.parquet',
    },
    split=['train', 'test'])
assert len(raw_train) == 25000 and len(raw_test) == 25000
assert raw_train.features['label'].names == ['neg', 'pos']

# Shuffle BEFORE splitting: the mirror's rows are ordered by label.
# Split the official training set 80/20, then use bounded development samples.
# The official test set stays separate and keeps all 25,000 labeled reviews.
development = raw_train.shuffle(seed=SEED)
split_at = int(0.8 * len(development))
MAX_TRAIN = 8000
MAX_VAL = 2000
MAX_LENGTH = 200
assert MAX_TRAIN <= split_at and MAX_VAL <= len(development) - split_at
train_rows = development.select(range(MAX_TRAIN))
val_rows = development.select(range(split_at, split_at + MAX_VAL))

# Convert to the same (text, label) tf.data interface used by the remaining lab.
def as_tf_dataset(rows):
    return tf.data.Dataset.from_tensor_slices(
        (list(rows['text']), np.asarray(rows['label'], dtype=np.int64)))

train_dataset = as_tf_dataset(train_rows)
val_dataset = as_tf_dataset(val_rows)
test_dataset = as_tf_dataset(raw_test)
train_labels = np.asarray(train_rows['label'])
val_labels = np.asarray(val_rows['label'])
majority_label = int(train_labels.mean() >= 0.5)
baseline_val_acc = float(np.mean(val_labels == majority_label))
print('Reviews used: train =', len(train_rows), ', validation =', len(val_rows),
      ', test =', len(raw_test))
print('Training label counts:', np.bincount(train_labels, minlength=2))
print('Validation label counts:', np.bincount(val_labels, minlength=2))
print('Majority label:', majority_label)
print('Baseline validation accuracy:', baseline_val_acc)
candidates = {}  # Store validation scores; never select a model using test scores.
train_dataset.element_spec

In [ ]:
# Return a dataset of (text, label) pairs
for example, label in train_dataset.take(3): # No. of examples
  print('Text: ', example.numpy())
  print('Label: ', label.numpy())

Next shuffle the training examples and create batches.

**Buffer size vs. batch size**
* The shuffle buffer is the pool of examples used to randomize order. It is separate from prefetching.
* Batch size is the number of examples used in one gradient update. Larger batches generally use more memory and give less noisy gradient estimates, although training and generalization depend on the setting.
* `prefetch` overlaps data preparation with model computation.

In [ ]:
BUFFER_SIZE = MAX_TRAIN
BATCH_SIZE = 64
train_dataset = train_dataset.shuffle(BUFFER_SIZE, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Print some examples from the shuffled dataset
for example, label in train_dataset.take(1):
  print('Texts: ', example.numpy()[:3])
  print()
  print('Labels: ', label.numpy()[:3])

##### **2. Building and Training the Model**

**A. Create the text encoder**

The raw text loaded by `tfds` needs to be processed before it can be used in a model. The simplest way to process text for training is using the `TextVectorization` layer. This layer has many capabilities, but this practice sticks to the default behavior.

The following code creates a text encoder using the `TextVectorization` layer in TensorFlow as follows (see <https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization> for details):

1. The `max_tokens` parameter sets the maximum size of the vocabulary to be used by the encoder. In this case, it is set to 1000.

1. The `adapt` method is called on the encoder object to build the vocabulary based on the training dataset. The `map` function is used to extract only the text data from the dataset, which is then passed to the `adapt` method. This allows the encoder to learn the vocabulary from the text data and create a mapping between words and integer indices.

1. After the vocabulary has been built, the encoder can be used to transform the text data into a sequence of integer indices that can be fed into a neural network.

In [ ]:
# Learn the vocabulary only from training text.
VOCAB_SIZE = 1000
encoder = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LENGTH)
encoder.adapt(train_dataset.map(lambda text, label: text))

The `.adapt` method sets the layer's vocabulary. Here are the first 20 tokens. After the padding and unknown tokens they're sorted by frequency:

In [ ]:
vocab = np.array(encoder.get_vocabulary())
vocab[:20]

The encoder returns integer token IDs, padded or truncated to `MAX_LENGTH = 200`. Zero denotes padding and one denotes an unknown token. This limit reduces training cost but can remove relevant information from long reviews. It stays fixed for both architectures.

In [ ]:
encoded_example = encoder(example)[:3].numpy()
encoded_example

The `[UNK]` token serves as a placeholder for words that are not found in the model's vocabulary during either training or inference.

In [ ]:
for n in range(3):
  print("Original: ", example[n].numpy())
  print("Round-trip: ", " ".join(vocab[encoded_example[n]]))
  print()

**B. Create the model**

![image](https://www.tensorflow.org/static/text/tutorials/images/bidirectional.png)

Above is a diagram of the model.

1. This model can be build as a `tf.keras.Sequential`.

2. The first layer is the `encoder`, which converts the text to a sequence of token indices.

3. After the encoder is an embedding layer. An embedding layer stores one vector per word. When called, it converts the sequences of word indices to sequences of vectors. These vectors are trainable. After training (on enough data), words with similar meanings often have similar vectors.

  This index-lookup is much more efficient than the equivalent operation of passing a one-hot encoded vector through a `tf.keras.layers.Dense` layer.

4. A recurrent neural network (RNN) processes sequence input by iterating through the elements. RNNs pass the outputs from one timestep to their input on the next timestep.

  The `tf.keras.layers.Bidirectional` wrapper can also be used with an RNN layer. This propagates the input forward and backwards through the RNN layer and then concatenates the final output.

  * A bidirectional RNN processes the sequence in both directions, making information from both ends available for the review representation.  

  * The main disadvantage of a bidirectional RNN is that you can't efficiently stream predictions as words are being added to the end.

5. After the RNN has converted the sequence to a single vector the two `layers.Dense` do some final processing, and convert from this vector representation to a single logit as the classification output.

The code to implement this is below:

In [ ]:
# Build the model
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=64,
        # Use masking to handle the variable sequence lengths
        mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])

**Masking and padding**
Padding adds placeholder token IDs so sequences share a common length. A mask marks those positions for compatible layers to ignore. Here, `Embedding(mask_zero=True)` supplies a mask to the LSTM. Truncation discards tokens beyond the chosen length.

**Consistent padding example**

Token-ID sequences:
```
[[71, 1331, 4231],
 [73, 8, 3215, 55, 927],
 [83, 91, 1, 645, 1253, 927]]
```
After post-padding to six positions:
```
[[71, 1331, 4231, 0, 0, 0],
 [73, 8, 3215, 55, 927, 0],
 [83, 91, 1, 645, 1253, 927]]
```
The integer tensor has shape `(batch_size, sequence_length)`. After embedding, the tensor has shape `(batch_size, sequence_length, embedding_dimension)`.

The embedding creates a padding mask. The LSTM consumes the sequence and mask and returns one vector per review. Later dense layers operate on this review vector.

In [ ]:
# Check if each layer supports masking
print([layer.supports_masking for layer in model.layers])

**C. Train the model**

In [ ]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.0)])

In [ ]:
history = model.fit(train_dataset, epochs=3,
    validation_data=val_dataset,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=1, restore_best_weights=True)])

In [ ]:
# Compare on validation data; keep test data unopened.
val_loss, val_acc = model.evaluate(val_dataset, verbose=0)
candidates['single_lstm'] = (model, float(val_acc))
print('Validation loss:', val_loss)
print('Validation accuracy:', val_acc)
print('Majority baseline validation accuracy:', baseline_val_acc)

In [ ]:
# Visualize the history of loss and accuracy by epoch
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plot_graphs(history, 'loss')
plt.ylim(None, 1)
plt.subplot(1, 2, 2)
plot_graphs(history, 'accuracy')
plt.ylim(0, None)

Run a prediction on a new sentence. The output is a **logit**. Positive logits correspond to probabilities above 0.5. We use `logit > 0` as the positive-label rule, consistent with the accuracy metric. Apply sigmoid to obtain the model's probability estimate.

In [ ]:
# TensorFlow string tensors avoid NumPy string-adapter compatibility issues.
# The next cell performs prediction.

In [ ]:
sample_text = 'The movie was cool. The animation and graphics were wonderful. I would recommend it.'
predictions = model.predict(tf.constant([sample_text]), verbose=0)
print('Logit:', float(predictions[0, 0]))
print('Positive probability:', float(tf.sigmoid(predictions)[0, 0]))
print('Predicted label:', int(predictions[0, 0] > 0))

In [ ]:
predictions

**D. Optional: stack two LSTM layers**

Skip this section if time or computing resources are limited. Architecture choices use validation scores only. Compare models trained with the same maximum epoch budget. The final cell selects between the baseline and whichever models you ran.

Keras recurrent layers have two available modes that are controlled by the `return_sequences` constructor argument:

* If `False` it returns only the last output for each input sequence (a 2D tensor of shape (`batch_size, output_features`)). This is the default, used in the previous model.

* If `True` the full sequences of successive outputs for each timestep is returned (a 3D tensor of shape `(batch_size, timesteps, output_features)`).

Here is what the flow of information looks like with `return_sequences=True`:

![image](https://www.tensorflow.org/static/text/tutorials/images/layered_bidirectional.png)

The interesting thing about using an RNN with `return_sequences=True` is that the output still has 3-axes, like the input, so it can be passed to another RNN layer, like this:

In [ ]:
# Define the RNN architecture
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(len(encoder.get_vocabulary()), 64, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1)
])

In [ ]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.0)])

In [ ]:
history = model.fit(train_dataset, epochs=3,
    validation_data=val_dataset,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=1, restore_best_weights=True)])

In [ ]:
val_loss, val_acc = model.evaluate(val_dataset, verbose=0)
candidates['stacked_lstm'] = (model, float(val_acc))
print('Stacked LSTM validation accuracy:', val_acc)
print('Majority baseline validation accuracy:', baseline_val_acc)

In [ ]:
# Optional architecture: inspect its training history.
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plot_graphs(history, 'loss')
plt.subplot(1, 2, 2)
plot_graphs(history, 'accuracy')

In [ ]:
# FINAL TEST: run once after all development choices are frozen.
# Exact ties prefer the baseline, then the simpler single-layer model.
selected_name, selected_score = 'majority_baseline', baseline_val_acc
selected_model = None
for name in ['single_lstm', 'stacked_lstm']:
    if name in candidates:
        candidate_model, score = candidates[name]
        if score > selected_score:
            selected_name, selected_score, selected_model = name, score, candidate_model
print('Validation-selected candidate:', selected_name)
test_labels = np.concatenate([y.numpy() for _, y in test_dataset])
baseline_test_acc = float(np.mean(test_labels == majority_label))
if selected_model is None:
    selected_test_acc = baseline_test_acc
else:
    _, selected_test_acc = selected_model.evaluate(test_dataset, verbose=0)
print('Baseline test accuracy:', baseline_test_acc)
print('Selected candidate test accuracy:', selected_test_acc)
# Report the result even if the selected neural model loses on test.
# Do not revise the model based on this test result.

**Reflection**
Did the validation-selected model beat the baseline on test? What information could truncation remove? Explain why a vocabulary adapted on all reviews would violate the development protocol. Training a small RNN here is different from extracting features from a pretrained Transformer.